# B1 · Titanic: a complete pipeline, three variants compared

**Type**: Implement with scikit-learn · **Level**: ●○○ guided · **Time**: about 40 minutes · **core path ★**

## Objective

Build with `ColumnTransformer` and `Pipeline` a model on Titanic that handles imputation, scaling and encoding by itself, evaluate it
in cross-validation, compare three preprocessing variants and read the coefficients with their names.

## Dataset

`ds.titanic()`: 1309 passengers, columns `pclass`, `sex`, `age` (20% missing), `siblings_spouses`, `parents_children`, `fare`, `port`,
`deck` (77% missing), `title` and the target `survived`. Stratified 75/25 split: `X_tr, X_te, y_tr, y_te`.

## Educational purpose

Put into practice the standard scheme of a tabular project: one preprocessor per group of columns, a pipeline, evaluation in
cross-validation on the training set and the test set touched only once.

## Task

1. Build `prep` (`ColumnTransformer`) with three groups: `num` (`NUM`: `SimpleImputer(median)` + `StandardScaler`), `cat` (`CAT`: `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown="ignore")`) and `pclass` (the column `pclass` with `"passthrough"`). Use `sparse_threshold=0` to get a dense array.
2. Build `model = make_pipeline(prep, LogisticRegression(max_iter=2000))` and save `cv_auc` (mean AUC in cross-validation `cv` on the training set).
3. Train on the training set and save `test_auc` (AUC on the test set). Save in `top5` the five feature names with the largest coefficient in absolute value (`model[:-1].get_feature_names_out()`).
4. Build `table` (rows: `base`, `without_deck`, `with_age_indicator`; column `cv_auc`) comparing three variants: the base one, one in which `deck` is removed from `CAT` and one in which the imputer of `age` has `add_indicator=True`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
def build(num=NUM, cat=CAT, age_indicator=False):
    num_pipe = ...
    cat_pipe = ...
    prep = ColumnTransformer([("num", num_pipe, num), ("cat", cat_pipe, cat), ("pclass", "passthrough", ["pclass"])],
                             sparse_threshold=0)
    return make_pipeline(prep, LogisticRegression(max_iter=2000))

In [ ]:
model = build()
cv_auc = ...
model.fit(X_tr, y_tr)
test_auc = ...
top5 = ...

In [ ]:
table = pd.DataFrame({"cv_auc": {
    "base": ...,
    "without_deck": ...,
    "with_age_indicator": ...,
}})
table.round(4)

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`num_pipe = make_pipeline(SimpleImputer(strategy="median", add_indicator=age_indicator), StandardScaler())`. For top5: `pd.Series(model[-1].coef_[0], index=model[:-1].get_feature_names_out()).abs().nlargest(5).index.tolist()`. For the variants: `cross_val_score(build(cat=[c for c in CAT if c != "deck"]), X_tr, y_tr, cv=cv, scoring="roc_auc").mean()`.'):
    check(0.80 < cv_auc < 0.90, "reasonable cross-validated AUC (between 0.80 and 0.90)", "with sex, title and pclass the AUC should exceed 0.8")
    check(0.80 < test_auc < 0.93, "reasonable AUC on the test set", "check that you trained on the training set and evaluated on the test set")
    check(len(top5) == 5 and any("sex" in n or "title" in n for n in top5), "sex or title appear among the five most influential features",
          "top5 = names sorted by |coefficient|, with the names from get_feature_names_out")
    check(set(table.index) == {"base", "without_deck", "with_age_indicator"} and table["cv_auc"].notna().all(), "table with the three variants",
          "rows: base, without_deck, with_age_indicator (in any order)")
    check(close(table.loc["base", "cv_auc"], cv_auc), "the base variant matches cv_auc", "the base variant is the model of step 2")
    check(abs(table.loc["without_deck", "cv_auc"] - table.loc["base", "cv_auc"]) < 0.02,
          "removing the deck (77% missing) changes the AUC little", "the deck is almost all missing: removing it changes the AUC by less than 0.02")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_titanic_complete_pipeline.ipynb`. Open it after you pass the check, or after you have really tried.